# Mesorch 训练（Kaggle · GPU T4 ×2）

在本机 `train_mesorch.sh`（4 卡）基础上改造成 **Kaggle 双 T4** 版本，并尽量补齐与论文
（AAAI 2025, Mesorch）的差距。

## 一、与论文的差距对照

| 论文配置 | 本机 3-epoch 冒烟 | 本 Notebook | 是否补齐 |
|---|---|---|---|
| SegFormer-B3 用 ImageNet 预训练 `mit_b3.pth` | 未加载（随机初始化） | 自动下载 / 挂载，并传 `--seg_pretrain_path` | ✅ |
| 有效 batch = 12 × 2 × 4 = **96** | 2 | 4 × 12 × 2 = **96** | ✅ |
| **150 epoch** | 3 | `EPOCHS = 150` + 断点续训 | ⚠️ 需 2~3 个 session |
| protocol-CAT 数据：CASIA2 + FantasticReality + IMD2020 + tampCOCO×4 | 仅 IMD2020 1800 张 | 自动扫描 `/kaggle/input`，拼出全部可用源 | ⚠️ **取决于你上传了哪些** |
| lr 1e-4 / min 5e-7 / wd 0.05 / warmup 2 / 512 | 部分 | 完全一致 | ✅ |
| ConvNeXt-Tiny 预训练 | 有 | 有 | ✅ |
| AMP | 关 | **开**（T4 无 bf16，走 fp16 + GradScaler） | ✅ 提速 |

> **数据集是唯一无法靠代码补齐的一项。** 第 5 节会打印"论文 7 个数据源里缺哪几个"。
> 缺得越多，F1 越没法跟论文 Table 1 比。至少把 **CASIA2.0** 和 **IMD2020** 传上去。

## 二、Kaggle 网页端准备

1. **Settings → Accelerator**：`GPU T4 ×2`；**Internet**：On（要 pip 装包、timm 下载 convnext_tiny）。
2. **Add Input → Datasets**，建议各建一个 Dataset 上传：
   - 训练数据：`CASIA2.0`（Tp/Gt）、`IMD2020_for_train`（Tp/Gt）、`FantasticReality_v1`、`tampCOCO`（4 个 json）
   - 预训练权重：`mit_b3.pth`（不传会自动下载，约 178 MB）
   - 代码：本仓库（不传会从 GitHub 克隆；仓库 404 时会报错提示你改用上传）
3. 只改「配置区」→ **Run All**。

> **时间预算**：Kaggle 单次 session 约 9~12 小时、每周约 30 GPU 小时。
> 建议先 `EPOCHS = 2` 跑通全流程，确认无误后设回 150，用 `AUTO_RESUME` 跨 session 续训。

## 0. 配置区（一般只改这里）

In [ ]:
# ============================== 配置区 ==============================
import os, sys, json, shutil, time, random, subprocess, urllib.request, re
from pathlib import Path

# ---------- 1. 代码来源 ----------
CODE_SOURCE = "auto"          # auto | github | input
REPO_URL    = "https://github.com/niumingcen/mesorch_new.git"
REPO_BRANCH = "main"

# ---------- 2. 训练数据 ----------
DATA_HINT = ""                # 留空 = 自动扫描 /kaggle/input；也可直接写某个数据集根目录
VAL_RATIO = 0.1               # 找不到论文测试集时，从训练集切这么多做验证

# ---------- 3. SegFormer-B3 预训练权重 ----------
MIT_B3_INPUT = ""             # 留空 = 自动找 mit_b3*.pth，找不到就下载
MIT_B3_URL   = "https://github.com/qubvel/segmentation_models.pytorch/releases/download/v0.0.2/mit_b3.pth"

# ---------- 4. 训练超参（对齐论文） ----------
MODEL            = "MesorchFull"   # MesorchFull(85.75M, 论文主模型) / Mesorch_P(62.24M, 剪枝版)
NPROC            = 2               # T4 ×2
BATCH_SIZE       = 4               # 每卡 batch
ACCUM_ITER       = 12              # 有效 batch = 4 × 12 × 2 = 96（论文 12 × 2 × 4 = 96）
TEST_BATCH_SIZE  = 4
EPOCHS           = 150             # 论文 150；单 session 跑不完就靠 AUTO_RESUME 续训
IMAGE_SIZE       = 512
LR               = 1e-4
MIN_LR           = 5e-7
WEIGHT_DECAY     = 0.05
WARMUP_EPOCHS    = 2
TEST_PERIOD      = 2
SAVE_PERIOD      = 10              # 普通 ckpt 每 N epoch 存一次（每个约 1 GB）
BEST_CKPT_MIN_EPOCH = 1            # 原代码硬编码 36，短程根本存不出 best
NUM_WORKERS      = 2
SEED             = 42
USE_AMP          = True            # T4 无 bf16 → fp16 + GradScaler；关掉慢 3~5 倍、显存翻倍
CONV_PRETRAIN    = True            # ConvNeXt-Tiny 用 timm 预训练（需联网）
AUTO_RESUME      = True            # 自动从 OUT_DIR 里最新的 ckpt 续训

# ---------- 5. 路径 / 环境 ----------
WORK_DIR = "/kaggle/working"
CODE_DIR = os.path.join(WORK_DIR, "Mesorch")
DATA_DIR = os.path.join(WORK_DIR, "data")
OUT_DIR  = os.path.join(WORK_DIR, "output_dir_" + MODEL.lower())
LOG_FILE = os.path.join(OUT_DIR, "train.log")
KEEP_CKPT = 2                      # 轮询时只保留最近 N 个 checkpoint-<epoch>.pth

HF_ENDPOINT = ""                   # HF 下不动就填 "https://hf-mirror.com"
NCCL_ENV = {"NCCL_DEBUG": "WARN"}  # NCCL 卡死时再加 NCCL_P2P_DISABLE / NCCL_SHM_DISABLE

os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
print("有效 batch =", BATCH_SIZE * ACCUM_ITER * NPROC, "（论文为 96）")
print("输出目录   =", OUT_DIR)

## 1. 环境体检

In [ ]:
import torch, multiprocessing
print("torch:", torch.__version__, "| torch.version.cuda:", torch.version.cuda)
n_gpu = torch.cuda.device_count()
print("GPU 数量:", n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print("  [%d] %s  %.1f GB  (sm_%d%d)" % (i, p.name, p.total_memory / 1024**3, p.major, p.minor))
if n_gpu and torch.cuda.get_device_properties(0).major < 8:
    print("注意: T4 是 Turing(sm_75)，无 bf16，AMP 走 fp16 + GradScaler —— 正常现象")
print("CPU 核数:", multiprocessing.cpu_count())
_, _, free = shutil.disk_usage("/kaggle/working")
print("/kaggle/working 剩余: %.1f GB" % (free / 1024**3))
if n_gpu < NPROC:
    print("!! 实际只有 %d 张 GPU，请把配置区 NPROC 改成 %d，并把 ACCUM_ITER 改成 %d 保持有效 batch=96"
          % (n_gpu, n_gpu, max(1, int(96 / (BATCH_SIZE * max(n_gpu, 1))))))

## 2. 安装依赖

Kaggle 镜像自带 torch，**不要重装 torch**（会把 CUDA 版本弄乱）。只补 `numpy<2`（IMDLBenCo 硬性要求）
与 `imdlbenco` / `timm`。

In [ ]:
def pip(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *pkgs]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd)
    print("  ->", "OK" if r.returncode == 0 else "FAILED(%d)" % r.returncode)
    return r.returncode

pip("numpy<2")              # sklearn / albumentations 按 numpy 1.x 编译，必须降
pip("imdlbenco>=0.1.45")
pip("-U", "timm")

In [ ]:
import numpy as np, importlib
print("numpy:", np.__version__, "（必须 < 2）")
for m in ["torch", "timm", "IMDLBenCo", "albumentations", "cv2", "fvcore", "tensorboard", "sklearn"]:
    try:
        mod = importlib.import_module(m)
        print("  %-16s %s" % (m, getattr(mod, "__version__", "?")))
    except Exception as e:
        print("  %-16s FAILED: %s: %s" % (m, type(e).__name__, e))

from IMDLBenCo.registry import MODELS, POSTFUNCS
from IMDLBenCo.datasets import ManiDataset, JsonDataset, BalancedDataset
from IMDLBenCo.evaluation import PixelF1
from IMDLBenCo.training_scripts.trainer import train_one_epoch
from IMDLBenCo.training_scripts.tester import test_one_epoch
print("IMDLBenCo 入口 import OK")

## 3. 拿到代码并打补丁

优先用挂载在 `/kaggle/input` 的代码，否则从 GitHub 克隆。随后做**定点**补丁（幂等）：
`--if_not_amp` 语义、`--best_ckpt_min_epoch`、`--save_period`。

In [ ]:
def find_input_code():
    base = Path("/kaggle/input")
    if not base.exists():
        return None
    for h in sorted(base.glob("*/train.py")) + sorted(base.glob("*/*/train.py")):
        if (h.parent / "mesorch.py").exists():
            return h.parent
    return None

src_dir = None
if CODE_SOURCE in ("auto", "input"):
    src_dir = find_input_code()
    if src_dir:
        print("使用挂载代码:", src_dir)

if src_dir is None and CODE_SOURCE != "input":
    ok = False
    if "<YOUR_USERNAME>" not in REPO_URL:
        if (Path(CODE_DIR) / ".git").exists():
            ok = subprocess.call(["git", "-C", CODE_DIR, "pull", "--ff-only", "origin", REPO_BRANCH]) == 0
        else:
            if Path(CODE_DIR).exists():
                shutil.rmtree(CODE_DIR)
            ok = subprocess.call(["git", "clone", "-b", REPO_BRANCH, "--depth", "1",
                                  REPO_URL, CODE_DIR]) == 0
    if not ok:
        raise RuntimeError(
            "无法从 GitHub 获取代码（仓库可能不存在/私有/分支名不对）。\n"
            "解决办法：把仓库代码作为 Dataset 上传到 /kaggle/input，并把配置区 CODE_SOURCE 设为 'input'")
    src_dir = Path(CODE_DIR)

if src_dir is None:
    raise RuntimeError("没找到代码，请检查 CODE_SOURCE / REPO_URL")

if Path(src_dir).resolve() != Path(CODE_DIR).resolve():
    if Path(CODE_DIR).exists():
        shutil.rmtree(CODE_DIR)
    shutil.copytree(src_dir, CODE_DIR)

if not (Path(CODE_DIR) / "train.py").exists():      # 代码可能在子目录
    for p in sorted(Path(CODE_DIR).glob("*/train.py")):
        CODE_DIR = str(p.parent); break

print("CODE_DIR =", CODE_DIR)
print(sorted(os.listdir(CODE_DIR)))

In [ ]:
tp = Path(CODE_DIR) / "train.py"
src = tp.read_text(encoding="utf-8")
out = src
notes = []

# (1) --if_not_amp: 默认要"开 AMP"，所以 action 必须是 store_true
m = re.search(r"parser\.add_argument\('--if_not_amp'.*?\)\n", out, re.S)
if m:
    if "store_false" in m.group(0):
        out = out[:m.start()] + m.group(0).replace("store_false", "store_true") + out[m.end():]
        notes.append("已修补 --if_not_amp -> store_true（默认开启 AMP）")
    else:
        notes.append("--if_not_amp 语义已正确")
else:
    notes.append("!! train.py 缺少 --if_not_amp，AMP 只能走默认")

# (2) --best_ckpt_min_epoch
if "--best_ckpt_min_epoch" not in out:
    anchor = "    parser.add_argument('--num_workers', default=1, type=int)\n"
    if anchor in out:
        out = out.replace(anchor, anchor +
            "    parser.add_argument('--best_ckpt_min_epoch', default=36, type=int,\n"
            "                        help='save best ckpt only when epoch >= this value')\n", 1)
        notes.append("已注入 --best_ckpt_min_epoch")
    else:
        notes.append("!! 找不到锚点，无法注入 --best_ckpt_min_epoch")

# (3) 硬编码的 epoch > 35
if "if epoch > 35:" in out:
    out = out.replace("if epoch > 35:", "if epoch >= args.best_ckpt_min_epoch:")
    notes.append("已把 'epoch > 35' 换成 args.best_ckpt_min_epoch")

# (4) --save_period
if "--save_period" not in out:
    anchor = "    parser.add_argument('--num_workers', default=1, type=int)\n"
    if anchor in out:
        out = out.replace(anchor, anchor +
            "    parser.add_argument('--save_period', default=2, type=int,\n"
            "                        help='save plain checkpoint every N epochs')\n", 1)
        notes.append("已注入 --save_period")
if "epoch % 2 == 0" in out:
    out = out.replace("epoch % 2 == 0", "epoch % args.save_period == 0")
    notes.append("已把 'epoch % 2 == 0' 换成 args.save_period")

if out != src:
    tp.write_text(out, encoding="utf-8")
for n in notes:
    print(" -", n)

AMP_PATCHED      = ("action='store_true'" in out) or ('action="store_true"' in out)
SUPPORT_SAVE     = "--save_period" in out
SUPPORT_BEST_MIN = "--best_ckpt_min_epoch" in out
SUPPORT_BEST_PTH = "checkpoint-best.pth" in out
print("")
print("AMP 默认开启:", AMP_PATCHED, "| --save_period:", SUPPORT_SAVE,
      "| --best_ckpt_min_epoch:", SUPPORT_BEST_MIN, "| checkpoint-best.pth:", SUPPORT_BEST_PTH)

## 4. SegFormer-B3 预训练权重（`mit_b3.pth`）

**这是补齐论文差距的关键一项**：Mesorch 的 Transformer 分支用 ImageNet 预训练的 MiT-B3 初始化。
本机那次 3-epoch 冒烟就是没加载它，F1 几乎为 0。

In [ ]:
PRETRAIN_DIR = os.path.join(WORK_DIR, "pretrain")
os.makedirs(PRETRAIN_DIR, exist_ok=True)

def find_first(pattern, root="/kaggle/input"):
    hits = sorted(Path(root).rglob(pattern))
    return str(hits[0]) if hits else None

seg_path = MIT_B3_INPUT or find_first("mit_b3*.pth") or find_first("mit_b3*.pt")
if seg_path and not os.path.isfile(seg_path):
    seg_path = None

if seg_path is None:
    seg_path = os.path.join(PRETRAIN_DIR, "mit_b3.pth")
    if not os.path.exists(seg_path):
        print("未找到本地 mit_b3，开始下载（约 178 MB）...")

        def _dl(url, dst):
            with urllib.request.urlopen(url) as r, open(dst, "wb") as f:
                total = int(r.headers.get("Content-Length", 0)); done = 0
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk:
                        break
                    f.write(chunk); done += len(chunk)
                    print("\r  %.0f / %.0f MB" % (done / 1024**2, total / 1024**2), end="")
            print()

        _dl(MIT_B3_URL, seg_path)

print("seg_pretrain_path =", seg_path, "(%.1f MB)" % (os.path.getsize(seg_path) / 1024**2))
assert os.path.getsize(seg_path) > 50 * 1024**2, "mit_b3.pth 体积异常，可能下载失败"

## 5. 数据集：拼出论文的 protocol-CAT 组合

自动扫描 `/kaggle/input`，把找到的数据源拼成一个 `BalancedDataset` 列表（等价于仓库里的
`balanced_dataset.json`），并打印**论文 7 个数据源的覆盖情况**。

> 目录型数据（含 `Tp/` + `Gt/`）会生成**显式配对 json**，而不是直接用 `ManiDataset`——
> `ManiDataset` 是按 `os.listdir` 的索引配对的，listdir 顺序不确定，有错位风险。

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def match_gt(stem, gt_map):
    for c in (stem, stem + "_mask", stem + "_gt", stem + "_m"):
        if c in gt_map:
            return gt_map[c]
    for k, v in gt_map.items():          # 兜底：前缀互含
        if k.startswith(stem) or stem.startswith(k):
            return v
    return None

def build_pairs_json(src_root, out_path, tag):
    tp, gt = src_root / "Tp", src_root / "Gt"
    gt_map = {g.stem.lower(): g for g in gt.iterdir() if g.suffix.lower() in IMG_EXT}
    pairs, unmatched = [], []
    for f in sorted(tp.iterdir()):
        if f.suffix.lower() not in IMG_EXT:
            continue
        g = match_gt(f.stem.lower(), gt_map)
        if g is None:
            unmatched.append(f.name); continue
        pairs.append([str(f), str(g)])
    json.dump(pairs, open(out_path, "w"))
    print("  %-22s 配对 %5d 对，未匹配 %d %s" % (tag, len(pairs), len(unmatched), unmatched[:3]))
    return pairs

def is_test_name(name):
    n = name.lower()
    return any(k in n for k in ("casia1", "casia_v1", "casia v1", "coverage", "nist", "columbia"))

def is_pairlist(d):
    return (isinstance(d, list) and d and isinstance(d[0], list) and len(d[0]) == 2
            and all(isinstance(x, str) for x in d[0]))

root = Path(DATA_HINT) if DATA_HINT else Path("/kaggle/input")
dir_ds, json_pair, json_balanced = [], [], []
if root.exists():
    for p in sorted(root.rglob("*")):
        if p.is_dir() and (p / "Tp").is_dir() and (p / "Gt").is_dir():
            dir_ds.append(p)
    for p in sorted(root.rglob("*.json")):
        try:
            d = json.loads(p.read_text(errors="ignore"))
        except Exception:
            continue
        if not (isinstance(d, list) and d):
            continue
        if is_pairlist(d):
            json_pair.append(p)
        elif isinstance(d[0], list) and d[0] and d[0][0] in ("ManiDataset", "JsonDataset"):
            json_balanced.append(p)

print("扫描到：目录型数据集 %d 个 | 配对 json %d 个 | 多源清单 json %d 个"
      % (len(dir_ds), len(json_pair), len(json_balanced)))
for p in dir_ds:        print("   dir :", p)
for p in json_pair:     print("   json:", p)
for p in json_balanced: print("   list:", p)

In [ ]:
train_entries, test_items = [], []
for p in dir_ds:
    if is_test_name(p.name):
        test_items.append(p)
    else:
        j = str(Path(DATA_DIR) / ("pairs_%s.json" % p.name))
        pairs, _ = build_pairs_json(p, j, p.name)
        if pairs:
            train_entries.append(["JsonDataset", j])
for p in json_pair:
    if is_test_name(p.name):
        test_items.append(p)
    else:
        train_entries.append(["JsonDataset", str(p)])

print("")
print("训练源:", [e[1].split("/")[-1] for e in train_entries] or "无")
print("测试源:", [str(p).split("/")[-1] for p in test_items] or "无")

# ---------- 论文 protocol-CAT 覆盖情况 ----------
EXPECTED = {
    "CASIA2.0": ("casia2",), "IMD2020": ("imd2020", "imd"),
    "FantasticReality": ("fantastic",), "tampCOCO-sp": ("sp_coco",),
    "tampCOCO-cm": ("cm_coco",), "tampCOCO-bcm": ("bcm_coco",), "tampCOCO-bcmc": ("bcmc_coco",),
}
joined = " ".join([p.name.lower() for p in dir_ds] +
                  [p.name.lower() for p in json_pair] +
                  [p.name.lower() for p in json_balanced]).lower()
missing = [k for k, keys in EXPECTED.items() if not any(x in joined for x in keys)]
print("")
print("=== 论文 protocol-CAT 覆盖 ===")
for k in EXPECTED:
    print("  %-18s %s" % (k, "OK" if k not in missing else "缺失"))
if missing:
    print("")
    print("!! 缺失 %d 个源：%s" % (len(missing), missing))
    print("!! 单一数据集 + 少量 epoch 训出来的 F1 无法与论文 Table 1 对比（论文是 7 源混合 + 150 epoch）。")
    print("!! 请至少补传 CASIA2.0 和 IMD2020。")

assert train_entries, "没有可用训练数据：/kaggle/input 下没找到含 Tp/ + Gt/ 的目录，也没有配对 json"

In [ ]:
# ---------- 训练集清单 ----------
if len(train_entries) == 1 and not json_balanced:
    TRAIN_LIST = train_entries[0][1]          # 单源：直接用配对 json
else:
    TRAIN_LIST = str(Path(DATA_DIR) / "balanced_train.json")
    entries = list(train_entries)
    for p in json_balanced:                  # 把现成的多源清单展开进去
        try:
            for e in json.loads(p.read_text()):
                if e not in entries:
                    entries.append(e)
        except Exception:
            pass
    json.dump(entries, open(TRAIN_LIST, "w"), indent=2)
print("TRAIN_LIST =", TRAIN_LIST)
print(open(TRAIN_LIST).read()[:400])

# ---------- 验证集 ----------
if test_items:                               # 优先用论文的测试集（CASIAv1/Coverage/NIST16/Columbia）
    t = test_items[0]
    if t.is_dir():
        VAL_LIST = str(Path(DATA_DIR) / ("pairs_val_%s.json" % t.name))
        build_pairs_json(t, VAL_LIST, "val/" + t.name)
    else:
        VAL_LIST = str(t)
    print("验证集 = 论文测试集:", VAL_LIST)
else:
    all_pairs = []
    for e in train_entries:
        try:
            all_pairs += json.loads(open(e[1]).read())
        except Exception:
            pass
    random.Random(SEED).shuffle(all_pairs)
    n_val = max(1, int(len(all_pairs) * VAL_RATIO))
    VAL_LIST = str(Path(DATA_DIR) / "pairs_val_split.json")
    json.dump(all_pairs[:n_val], open(VAL_LIST, "w"))
    print("!! 没找到论文测试集，改为从训练集切 %d 张做验证（F1 只能看趋势，不能与论文比）" % n_val)
print("VAL_LIST =", VAL_LIST)

In [ ]:
# ---------- 配对自检：抽 3 条确认图像/掩膜尺寸一致、掩膜非空 ----------
import cv2
try:
    head = json.loads(open(TRAIN_LIST).read())
    samples = head if is_pairlist(head) else json.loads(open(head[0][1]).read())
except Exception as e:
    samples = []
    print("自检跳过:", e)
for img_p, msk_p in samples[:3]:
    im = cv2.imread(img_p, cv2.IMREAD_COLOR)
    mk = cv2.imread(msk_p, cv2.IMREAD_GRAYSCALE)
    if im is None or mk is None:
        print("  !! 读不到:", img_p, msk_p); continue
    pos = float((mk > 127).mean())
    ok = (im.shape[0] == mk.shape[0] and im.shape[1] == mk.shape[1] and pos > 0)
    print("  %-30s img=%s mask=%s 正样本占比=%.3f %s"
          % (Path(img_p).name, im.shape[:2], mk.shape[:2], pos, "" if ok else " <-- 异常"))

## 6. 单卡冒烟（强烈建议先跑）

验证：mit_b3 能加载、模型能前向/反向、**峰值显存**够不够。会顺带从 timm 下载 `convnext_tiny`，
同时验证 Internet 可用。看峰值显存再决定 `BATCH_SIZE`。

In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    os.chdir(CODE_DIR)
    if CODE_DIR not in sys.path:
        sys.path.insert(0, CODE_DIR)
    mesorch_mod   = importlib.import_module("mesorch")
    mesorch_p_mod = importlib.import_module("mesorch_p")
    cls = getattr(mesorch_mod, MODEL, None) or getattr(mesorch_p_mod, MODEL)

    torch.cuda.empty_cache()
    model = cls(seg_pretrain_path=seg_path, conv_pretrain=CONV_PRETRAIN, image_size=IMAGE_SIZE).cuda()
    print("%s 参数量: %.2f M" % (MODEL, sum(p.numel() for p in model.parameters()) / 1e6))

    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    img = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device="cuda")
    msk = torch.zeros(BATCH_SIZE, 1, IMAGE_SIZE, IMAGE_SIZE, device="cuda")

    torch.cuda.reset_peak_memory_stats()
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        out = model(img, msk)
        loss = out["backward_loss"]
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update()

    print("loss = %.4f | pred_mask = %s" % (float(loss), tuple(out["pred_mask"].shape)))
    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("峰值显存: %.2f GB / %.1f GB" % (peak, total))
    print("提示: 峰值超过 13 GB 就把 BATCH_SIZE 调小，并把 ACCUM_ITER 按比例调大，"
          "保持 BATCH_SIZE × ACCUM_ITER × %d = 96" % NPROC)

    del model, opt, img, msk, out, loss, scaler
    torch.cuda.empty_cache()

## 7. 组装命令并启动训练

In [ ]:
def latest_ckpt():
    if not Path(OUT_DIR).exists():
        return None
    best = Path(OUT_DIR) / "checkpoint-best.pth"
    def ep(p):
        s = p.stem.split("-")[-1]
        return int(s) if s.isdigit() else -1
    cands = sorted([p for p in Path(OUT_DIR).glob("checkpoint-*.pth") if "best" not in p.name], key=ep)
    if best.exists():
        return str(best)
    return str(cands[-1]) if cands else None

RESUME_CKPT = latest_ckpt() if AUTO_RESUME else None
print("续训权重:", RESUME_CKPT if RESUME_CKPT else "（无，从头训练）")

In [ ]:
def q(a):
    a = str(a)
    return '"%s"' % a if " " in a else a

cmd = [
    "torchrun", "--standalone", "--nnodes=1", "--nproc_per_node=%d" % NPROC, "./train.py",
    "--model", MODEL,
    "--conv_pretrain", str(CONV_PRETRAIN),
    "--seg_pretrain_path", seg_path,
    "--world_size", str(NPROC),
    "--find_unused_parameters",          # 必须：ConvNeXt 的 head 未参与 loss
    "--batch_size", str(BATCH_SIZE),
    "--test_batch_size", str(TEST_BATCH_SIZE),
    "--data_path", TRAIN_LIST,
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--image_size", str(IMAGE_SIZE),
    "--if_resizing",
    "--min_lr", str(MIN_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--test_data_path", VAL_LIST,
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--output_dir", OUT_DIR,
    "--log_dir", OUT_DIR,
    "--accum_iter", str(ACCUM_ITER),
    "--seed", str(SEED),
    "--test_period", str(TEST_PERIOD),
    "--num_workers", str(NUM_WORKERS),
]

# 语义：修补后 --if_not_amp 是 store_true（默认 False = 开 AMP）
if not USE_AMP and AMP_PATCHED:
    cmd.append("--if_not_amp")
elif USE_AMP and not AMP_PATCHED:
    cmd.append("--if_not_amp")           # 旧语义：传了才是开 AMP
    print("警告: 旧版 --if_not_amp 语义，只能靠显式传参开启 AMP")

if SUPPORT_SAVE:
    cmd += ["--save_period", str(SAVE_PERIOD)]
else:
    print("提示: 该 train.py 不支持 --save_period（默认每 2 epoch 存一次），轮询会自动清理旧 ckpt")

if SUPPORT_BEST_MIN:
    cmd += ["--best_ckpt_min_epoch", str(BEST_CKPT_MIN_EPOCH)]

if RESUME_CKPT:
    cmd += ["--resume", RESUME_CKPT]

env_lines = [
    "export CUDA_VISIBLE_DEVICES=%s" % ",".join(str(i) for i in range(NPROC)),
    "export OMP_NUM_THREADS=4",
    "export TOKENIZERS_PARALLELISM=false",
]
for k, v in NCCL_ENV.items():
    env_lines.append("export %s=%s" % (k, v))
if HF_ENDPOINT:
    env_lines.append("export HF_ENDPOINT=%s" % HF_ENDPOINT)

sh_text = "#!/usr/bin/env bash\nset -x\ncd %s\n%s\n\n%s 2>&1\n" % (
    q(CODE_DIR), "\n".join(env_lines), " \\\n  ".join(q(c) for c in cmd))
sh_path = os.path.join(WORK_DIR, "run_train.sh")
Path(sh_path).write_text(sh_text)
print(sh_text)

In [ ]:
log_f = open(LOG_FILE, "ab", buffering=0)
env = os.environ.copy()
env.update(NCCL_ENV)
env["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in range(NPROC))
env["OMP_NUM_THREADS"] = "4"
env["TOKENIZERS_PARALLELISM"] = "false"
if HF_ENDPOINT:
    env["HF_ENDPOINT"] = HF_ENDPOINT

TRAIN_PROC = subprocess.Popen(["bash", sh_path], stdin=subprocess.DEVNULL,
                              stdout=log_f, stderr=subprocess.STDOUT,
                              env=env, cwd=CODE_DIR, start_new_session=True)
print("训练已启动, PID =", TRAIN_PROC.pid)
print("日志文件:", LOG_FILE)

time.sleep(90)
txt = Path(LOG_FILE).read_text(errors="ignore") if Path(LOG_FILE).exists() else "(暂无日志)"
print("\n".join(txt.strip().split("\n")[-40:]))

### 7.1 跟进度（可反复运行；中断本 cell 不会杀掉训练）

In [ ]:
POLL_INTERVAL = 120     # 秒
POLL_TIMES    = 240     # 240 × 120s ≈ 8 小时

def cleanup_ckpts(keep=KEEP_CKPT):
    out = Path(OUT_DIR)
    if not out.exists():
        return
    def _epoch(p):
        s = p.stem.split("-")[-1]
        return int(s) if s.isdigit() else -1
    ck = sorted([p for p in out.glob("checkpoint-*.pth") if "best" not in p.name], key=_epoch)
    for p in ck[:-keep]:
        try:
            p.unlink(); print("  清理旧 ckpt:", p.name)
        except Exception as e:
            print("  清理失败:", p.name, e)

for i in range(POLL_TIMES):
    time.sleep(POLL_INTERVAL)
    txt = Path(LOG_FILE).read_text(errors="ignore") if Path(LOG_FILE).exists() else ""
    print("=" * 22, time.strftime("%H:%M:%S"), "pid=%s" % TRAIN_PROC.poll(), "=" * 22)
    print("\n".join(txt.strip().split("\n")[-25:]))
    cleanup_ckpts()
    if TRAIN_PROC.poll() is not None:
        print("训练进程已结束, returncode =", TRAIN_PROC.returncode)
        break
else:
    print("达到轮询上限，训练可能仍在跑；重跑本 cell 继续查看")

In [ ]:
# 需要中断训练时运行这一格
try:
    TRAIN_PROC.terminate()
    print("已发送 SIGTERM 给 PID", TRAIN_PROC.pid, "当前状态:", TRAIN_PROC.poll())
except NameError:
    print("还没有启动训练进程")

## 8. 结果与跨 session 续训

Kaggle 单次 session 装不下 150 epoch。session 到期前：
1. 把 `/kaggle/working/output_dir_*` 存成 Kaggle Dataset（Save Version）；
2. 下次开新 session 把它挂进 `/kaggle/input`，把权重拷回 `OUT_DIR`，保持 `AUTO_RESUME = True` 即可接着训。

> 注意：`save_model` 只写 `checkpoint-<epoch>.pth` + `checkpoint-best.pth`，
> **没有单独的 best 标记文件**，最佳轮次以 `log.txt` 里的 F1 为准。

In [ ]:
print("输出目录:", OUT_DIR)
for p in sorted(Path(OUT_DIR).glob("*")):
    print("  %-28s %8.1f MB" % (p.name, p.stat().st_size / 1024**2))

best = None
log_txt = Path(OUT_DIR) / "log.txt"
if log_txt.exists():
    rows = []
    for line in log_txt.read_text(errors="ignore").splitlines():
        try:
            d = json.loads(line)
        except Exception:
            continue
        ks = [k for k in d if k.startswith("test_") and "f1" in k.lower()]
        if ks:
            rows.append((d.get("epoch"), d[ks[0]], d.get("train_predict_loss")))
    print("")
    print("epoch | F1       | train_loss")
    for e, f, l in rows:
        print("  %3s | %.5f | %s" % (e, f, ("%.4f" % l) if isinstance(l, (int, float)) else l))
    if rows:
        best = max(rows, key=lambda r: r[1])
print("")
print("最佳 (epoch, F1, loss):", best)

## 9. 常见问题

| 现象 | 处理 |
|---|---|
| `CUDA out of memory` | 调小 `BATCH_SIZE`，把 `ACCUM_ITER` 按比例调大（保持 `BATCH_SIZE × ACCUM_ITER × 2 = 96`）；`TEST_BATCH_SIZE` 降到 1~2 |
| DDP 报 "parameters that were not used in producing loss" | 命令里已带 `--find_unused_parameters`（ConvNeXt 的 head 未参与 loss） |
| 卡在 NCCL / `unhandled system error` | 配置区加 `"NCCL_P2P_DISABLE": "1"`，仍不行再加 `"NCCL_SHM_DISABLE": "1"`；实在不行 `NPROC=1` 单卡 |
| loss 变 NaN | 先 `USE_AMP = False` 排除 fp16 溢出；Mesorch 的 DCT 分支做 min-max 归一化，纯色图可能除零 |
| `UnboundLocalError: global_rank` | 说明没走 `torchrun`；直接 `python train.py` 会崩 |
| `ModuleNotFoundError: No module named 'mesorch'` | `CODE_DIR` 不对，检查第 3 步打印的目录里有没有 `mesorch.py` |
| `numpy.dtype size changed` / DLL 报错 | numpy 没降到 1.x，重跑 `pip("numpy<2")` 后**重启 kernel** |
| timm 下载 convnext 超时 | 配置区填 `HF_ENDPOINT = "https://hf-mirror.com"` |
| GitHub 克隆失败 | 仓库不存在/私有 → 把代码作为 Dataset 上传，配置区设 `CODE_SOURCE = "input"` |
| 输出容量超限 | 轮询 cell 会自动清理旧 ckpt，或把 `SAVE_PERIOD` 调更大 |
| F1 一直很低 | 先确认第 4 步 `mit_b3.pth` 真的加载了；再看第 5 步的数据源覆盖——只训单一数据集 + 少量 epoch 基本不可能复现论文数值 |